# 📖 Notebook 1: LangChain Agents
### Phase 3 – Advanced LangChain Concepts

---

## What is an Agent?

A **LangChain Agent** is an LLM that can **decide which actions to take** in order to answer a question.  
Unlike a regular LLM that just generates text, an agent can:

- 🔍 Search the web
- 🧮 Run Python code
- 📖 Query a database
- 🔁 Loop and revise its reasoning

**The key idea**: Give the LLM a set of *tools* and let it figure out *which ones to use* and *when*.

```
User Question
     │
     ▼
  [Agent]
     │── Decides to call: web_search("X")
     │── Gets back: search results
     │── Decides: "I have enough info"
     ▼
  Final Answer
```

---

## Agent Types in LangChain

| Agent Type | How it works | Best for |
|---|---|---|
| **Zero-shot** | Decides from tool descriptions alone, no examples | General tasks |
| **ReAct** | Reason → Act → Observe loop | Complex, multi-step tasks |
| **Self-ask** | Breaks into sub-questions | Compositional questions |
| **Conversational** | Maintains chat history | Chatbots |
| **Plan-and-Execute** | Plans all steps first, then executes | Long sequences |


## ReAct Agent – The Most Important Pattern

**ReAct = Reasoning + Acting**

The agent alternates between:
1. **Thought** – Reason about what to do next  
2. **Action** – Call a tool  
3. **Observation** – Read the tool result  
4. Repeat until → **Final Answer**

### Example Trace

```
Question: Who is the current president of France and how old are they?

Thought: I need to find the current president of France.
Action: web_search
Action Input: "current president of France 2024"
Observation: Emmanuel Macron is the current president of France.

Thought: Now I need his age.
Action: web_search
Action Input: "Emmanuel Macron age"
Observation: Emmanuel Macron was born on December 21, 1977.

Thought: I can calculate his age = 2024 - 1977 = 46.
Final Answer: Emmanuel Macron, born in 1977, is 46 years old.
```

This **transparent reasoning** makes ReAct agents easy to debug.


In [ ]:
# Setup – run this first
import os
from dotenv import load_dotenv

# Load .env from the parent directory
load_dotenv('../.env')

GROQ_API_KEY = os.getenv('GROQ_API_KEY')
print('API Key loaded:', '✅' if GROQ_API_KEY and GROQ_API_KEY != 'your_groq_api_key_here' else '❌ Not set')

In [ ]:
# ── Create a simple ReAct agent ──────────────────────────────────────────────
from langchain import hub
from langchain.agents import AgentExecutor, create_react_agent
from langchain_groq import ChatGroq
from langchain.tools import Tool
from langchain_community.tools import DuckDuckGoSearchRun

# 1. LLM
llm = ChatGroq(
    model='llama-3.3-70b-versatile',
    temperature=0.3,
    groq_api_key=GROQ_API_KEY
)

# 2. Tools
search = DuckDuckGoSearchRun()
tools = [
    Tool(
        name='web_search',
        func=search.run,
        description='Search the web. Input: a search query string.'
    )
]

# 3. Prompt (standard ReAct prompt from LangChain Hub)
prompt = hub.pull('hwchase17/react')

# 4. Create agent
agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)

# 5. Executor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,          # Show full Thought/Action/Observation trace!
    max_iterations=5,
    handle_parsing_errors=True,
)

print('✅ ReAct agent created successfully!')

In [ ]:
# ── Run the agent – watch the Thought/Action/Observation trace ───────────────
result = agent_executor.invoke({
    'input': 'What is LangChain and when was it created?'
})

print('\n' + '='*60)
print('FINAL ANSWER:')
print(result['output'])

## Zero-Shot vs ReAct

In modern LangChain, **`create_react_agent` is inherently zero-shot** — it receives no examples.  
The LLM decides which tool to use purely from reading the tool `description` strings.

**Key lesson**: Write clear, specific tool descriptions. They ARE the agent's instructions.

```python
# ❌ Vague description → agent won't use this correctly
Tool(name='search', func=..., description='Searches stuff')

# ✅ Clear description → agent knows exactly when/how to use it
Tool(
    name='web_search',
    func=...,
    description=(
        'Searches the web using DuckDuckGo. Use for current events, '
        'recent news, or facts not in training data. '
        'Input: a search query string.'
    )
)
```


## Key Concepts Summary

| Concept | Description |
|---|---|
| `create_react_agent` | Creates the agent brain |
| `AgentExecutor` | Runs the action loop |
| `Tool` | A callable the agent can use |
| `hub.pull(...)` | Downloads a pre-built prompt |
| `verbose=True` | Shows reasoning trace |
| `max_iterations` | Prevents infinite loops |
| `handle_parsing_errors` | Recovers from LLM format mistakes |

**👉 Next:** See `02_tools.ipynb` to learn how to build and integrate different tool types.
